In [ ]:
import os
import rasterio
from rasterio.windows import Window
import numpy as np
from tqdm import tqdm

# === Paths ===
input_dir = r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\LCMAP_edge_age_outputs"
output_dir = r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\LCMAP_edge_age_mean_1km"
os.makedirs(output_dir, exist_ok=True)

# === Parameters ===
directions = ['north', 'south', 'east', 'west']
years = range(2021, 2022)  # You can extend this range
scale = 33  # ~990m resolution

# === Loop over years ===
for year in years:
    input_paths = [os.path.join(input_dir, f"EdgeAge_{d}_{year}.tif") for d in directions]
    
    # Check for missing files
    if not all(os.path.exists(p) for p in input_paths):
        print(f"Skipping {year} due to missing edge age files.")
        continue

    # Open all four directions
    srcs = [rasterio.open(p) for p in input_paths]
    profile = srcs[0].profile
    transform = srcs[0].transform
    height, width = srcs[0].height, srcs[0].width

    # === Prepare output raster profile ===
    new_height = height // scale
    new_width = width // scale
    new_transform = rasterio.Affine(
        transform.a * scale, transform.b, transform.c,
        transform.d, transform.e * scale, transform.f
    )
    profile.update(
        dtype='float32',
        height=new_height,
        width=new_width,
        transform=new_transform,
        compress='lzw',
        count=1
    )
    out_path = os.path.join(output_dir, f"EdgeAge_mean_{year}_990m.tif")
    
    with rasterio.open(out_path, 'w', **profile) as dst:
        for i in tqdm(range(new_height), desc=f"Year {year}", leave=False):
            for j in range(new_width):
                row_off = i * scale
                col_off = j * scale
                h = min(scale, height - row_off)
                w = min(scale, width - col_off)
                win = Window(col_off, row_off, w, h)

                stack = []

                for src in srcs:
                    data = src.read(1, window=win).astype(np.float32)
                    data[data == 0] = np.nan  # Mask non-edges

                    stack.append(data)

                stack = np.stack(stack)

                # === Weighted average based on valid pixels ===
                valid_mask = ~np.isnan(stack)
                total_sum = np.nansum(stack)
                total_count = np.sum(valid_mask)

                if total_count == 0:
                    block_mean = np.nan
                else:
                    block_mean = total_sum / total_count

                dst.write(np.array([[block_mean]], dtype=np.float32), 1, window=Window(j, i, 1, 1))

    # Close input files
    for src in srcs:
        src.close()